In [0]:
from pyspark.sql.functions import (col,upper,trim,concat_ws,coalesce,lit,udf,when,max,create_map,to_timestamp,date_sub,to_date,first,lower,size,split)
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
from pyspark.sql import Row
from pyspark.sql import functions as F

In [0]:
tables=spark.catalog.listTables("f1_warehouse.gold")

gold={}


for table in tables:
    df=spark.table(f"f1_warehouse.gold.{table.name}")
    gold[table.name]=df


In [0]:
for name,df in gold.items():
    print(name)
   

## Data Check functions

* not null 
* referential integrity: values form fact table that hav no matching value in the dimension table
* duplicate checks
* range checks 

In [0]:
# complete cols: cannot be null values in the col
# unique cols: no duplicate values in the col
# date cols: should be in date format


def data_quality_checks(df,table_name,complete_cols=None,unique_cols=None,date_cols=None,foreign_keys=None):

    results=[] #stores a summary for every check
    errors={} #labelled folders for every error

#null checks
    if complete_cols:
        for col_name in complete_cols:
            is_null_df=df.filter(col(col_name).isNull())
            count=is_null_df.count()

            results.append({
                "table":table_name,
                "check":"NOT_NULL",
                "column":col_name,
                "failed_rows":count,
                "status":"FAIL" if count>0 else "PASS"
            })

            if count>0:
                errors[f"{table_name}_{col_name}_null"]=is_null_df

#unique checks
    if unique_cols:
        for col_name in unique_cols:
            is_duplicate=df.groupBy(col_name).count().filter(col("count")>1)
            count=is_duplicate.count()

            results.append({
                "table":table_name,
                "check":"UNIQUE",
                "column":col_name,
                "failed_rows":count,
                "status":"FAIL" if count>0 else "PASS"
            })

            if count>0:
                errors[f"{table_name}_{col_name}_duplicate"]=is_duplicate

#date checks
    if date_cols:
        for col_name in date_cols:
            is_invalid=df.filter(col(col_name).cast("date").isNull())
            count=is_invalid.count()

            results.append({
                "table":table_name,
                "check":"DATE",
                "column":col_name,
                "failed_rows":count,
                "status":"FAIL" if count>0 else "PASS"
            })

            if count>0:
                errors[f"{table_name}_{col_name}_date_invalid"]=is_invalid

# referential integrity checks
    if foreign_keys:
        for foreign_key,(reference_df,reference_col) in foreign_keys.items():
            
            error_df=(df.select(foreign_key).distinct().
                        join(
                            reference_df.select(
                                col(reference_col).alias(foreign_key)
                            ).distinct(),
                            on=foreign_key,
                            how="left_anti" # what in the left table doesnt exist in the right table
                        )
            )

            count=error_df.count()

            results.append({
                "table":table_name,
                "check":"FOREIGN_KEY",
                "column":foreign_key,
                "failed_rows":count,
                "status":"FAIL" if count>0 else "PASS"})

            if count>0:
                errors[f"{table_name}_{foreign_key}_foreign_key"]=error_df

    return results,errors




### Check circuits dim table

In [0]:
results_circuits,errors_circuits=data_quality_checks(gold["dim_circuits"], "dim_circuits", complete_cols=["circuit_id","location"], unique_cols=["circuit_id"])
display(results_circuits)

### Check driver dim table

In [0]:
results_drivers,errors_drivers=data_quality_checks(gold["dim_driver"],"dim_driver",complete_cols=["driver_id","driver_ref"],unique_cols=["driver_id"],date_cols=["dob"])
display(results_drivers)

In [0]:
gold["dim_driver"].show(5)

In [0]:
gold["dim_driver"].filter(col("code") == "MAG").show()

### Check teams dim table

In [0]:
results_teams,errors_teams=data_quality_checks(gold["dim_teams"],"dim_teams",complete_cols=["constructor_id","team_name"],unique_cols=["constructor_id","team_name"])
display(results_teams)

### Check meetings dim table

In [0]:
results_meetings,errors_meetings=data_quality_checks(gold["dim_meetings"],"dim_meetings",complete_cols=["race_meeting_id","circuit_id","meeting_name"],unique_cols=["race_meeting_id"],date_cols=["date_start","date_end"])
display(results_meetings)

### Check fact pit stop tables 

In [0]:
gold["fact_pit_stops"].show(5)

In [0]:
gold["fact_pit_stops"].count()

In [0]:
results_pit_stops,errors_pit_stops=data_quality_checks(
    gold["fact_pit_stops"],
    "fact_pit_stops",
    complete_cols=[
        "driver_id",
        "race_meeting_id"
    ],
    date_cols=["stop_time"],
    foreign_keys={
        "driver_id": (gold["dim_driver"], "driver_id"),
        "race_meeting_id": (gold["dim_meetings"], "race_meeting_id")
    }
)

display(results_pit_stops)

In [0]:
for table_name,df in errors_pit_stops.items():
    print(table_name)


In [0]:
errors_pit_stops["fact_pit_stops_driver_id_null"].select("race_meeting_id").distinct().show()
errors_pit_stops["fact_pit_stops_stop_number_null"].select("race_meeting_id").distinct().show()
errors_pit_stops["fact_pit_stops_stop_duration_null"].select("race_meeting_id").distinct().show()


### Check fact race results table

In [0]:
gold["fact_race_results"].show(5)

In [0]:
results_race,errors_race=data_quality_checks(
    gold["fact_race_results"],
    "fact_race_results",
    complete_cols=[
        "race_meeting_id",
        "driver_id",
        "constructor_id"],
    foreign_keys={
        "race_meeting_id": (gold["dim_meetings"], "race_meeting_id"),
        "driver_id": (gold["dim_driver"],"driver_id"),
        "constructor_id": (gold["dim_teams"],"constructor_id")
    }
)
display(results_race)


In [0]:
for table_name,df in errors_race.items():
    print(table_name)

In [0]:
errors_race["fact_race_results_position_null"].show(10)

In [0]:
#no of disqualified/did not start/did not finish positions (which cld explain why position is null)
errors_race["fact_race_results_position_null"].filter(col("dnf")|col("dns")|col("dsq")).count()

In [0]:
# remaining 29 rows
errors_race["fact_race_results_position_null"].filter(~(col("dnf") | col("dns") | col("dsq"))).show()


* 11111 out of 11140 cols have no positons due to drivers dnf/dns/dnq
* remaining 29 is either due to dnf/nc (not classified) but we'll create a nc col to categorise these remaining 29 vals


### Check fact qualifying table

In [0]:
gold["fact_qualifying"].show(5)

In [0]:
results_qualifying,errors_qualifying=data_quality_checks(
    gold["fact_qualifying"],
    "fact_qualifying",
    complete_cols=[
        "race_meeting_id",
        "driver_id",
        "constructor_id"],
    foreign_keys={
        "race_meeting_id": (gold["dim_meetings"], "race_meeting_id"),
        "driver_id": (gold["dim_driver"],"driver_id"),
        "constructor_id": (gold["dim_teams"],"constructor_id")
    }
)
display(results_qualifying)


In [0]:
# view mismatch in constructor_id 

gold["fact_qualifying"].select("constructor_id").distinct().join(
    gold["dim_teams"].select("constructor_id").distinct(),
    on="constructor_id",
    how="leftanti"
).show()

In [0]:
for name in errors_qualifying:
    print(name)

In [0]:
errors_qualifying["fact_qualifying_constructor_id_null"].groupBy("driver_id").count().filter(col("count")>1).show()

In [0]:
# replace constuctor id value for driver_id 866

In [0]:
#